In [37]:
%matplotlib tk

import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import detectors as det
import filters as filt

csv_file_name = 'initial_walk_test_10-08-2026_16-31-45_3.csv'
csv_path = '../data/measured_walks/'
csv_save_path = '../data/orientation_mahony/'
df = pd.read_csv(f'{csv_path}{csv_file_name}')

In [38]:
# =============
# Set parameters and detect ZVWs
# =============
acc_dev = det.ACC_DEVIATION
gyro_limit = det.GYRO_LIMIT
var_limit = det.VAR_LIMIT
var_window = det.VAR_WINDOW
dwell = det.DWELL

ax = df['ax']
ay = df['ay']
az = df['az']
gx = df['gx']
gy = df['gy']
gz = df['gz']

dt_array = np.diff(df['t_us'] - df['t_us'].iloc[0]) / 1e6
# Add a mean value at the beginning
dt_array = np.insert(dt_array, 0, dt_array.mean())

zvw_mask = det.detect_zvw(df, acc_dev, gyro_limit, var_limit, var_window, dwell)
masked_quats = filt.mahony_filter(ax, ay, az, gx, gy, gz, dt_array, zvw_mask)
masked_roll, masked_pitch, masked_yaw = filt.quaternions_to_euler(masked_quats)

no_zvw_mask = np.zeros_like(zvw_mask, dtype=bool)
unmasked_quats = filt.mahony_filter(ax, ay, az, gx, gy, gz, dt_array, no_zvw_mask)
unmasked_roll, unmasked_pitch, unmasked_yaw = filt.quaternions_to_euler(unmasked_quats)


In [39]:
# =============
# Plot Roll and pitch
# =============

time_sec = (df['t_us'] - df['t_us'].iloc[0]) / 1e6

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

ax1.plot(time_sec, unmasked_roll, label='Gyro Only (Drifting)', color='tab:red', alpha=0.6, linewidth=1)
ax1.plot(time_sec, masked_roll, label='ZVW Gated Mahony (Stable)', color='tab:blue', linewidth=1.5)
ax1.set_title('Roll (X-Axis Tilt): Gyro Drift vs. ZVW Gated Mahony')
ax1.set_ylabel('Angle (deg)')
ax1.set_xlabel('Time (s)')
ax1.legend(loc='upper left')
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.plot(time_sec, unmasked_pitch, label='Gyro only (drifting)', color='tab:red', alpha=0.6, linewidth=1)
ax2.plot(time_sec, masked_pitch, label='ZVW Gated Mahony (Stable)', color='tab:blue', linewidth=1.5)
ax2.set_title('Pitch (Y-Axis Tilt): Gyro Drift vs. ZVW Gated Mahony')
ax2.set_ylabel('Angle (deg)')
ax2.set_xlabel('Time (s)')
ax2.legend(loc='upper left')
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_roll_pitch.png'), dpi=120)
plt.show()


In [40]:
# =============
# Plot ZVW Residuals
# =============

raw_accel = np.column_stack((ax, ay, az))

# Rotate the raw acceleration into the global frame
global_accel = filt.rotate_vector_by_quaternion(raw_accel, masked_quats)

linear_accel = np.copy(global_accel)
linear_accel[:, 2] -= 1.0

zvs_residuals = linear_accel[zvw_mask]

mean_x_error = np.mean(zvs_residuals[:, 0])
mean_y_error = np.mean(zvs_residuals[:, 1])
mean_z_error = np.mean(zvs_residuals[:, 2])

print(f"Mean X Error: {mean_x_error} g")
print(f"Mean Y Error: {mean_y_error} g")
print(f"Mean Z Error: {mean_z_error} g")

# Histogram Z Error
fig, ax_hist = plt.subplots(1, 1, figsize=(8, 4))
ax_hist.hist(zvs_residuals[:, 0], bins=np.arange(-0.5, 0.5, 0.01), label='X Error')
ax_hist.hist(zvs_residuals[:, 1], bins=np.arange(-0.5, 0.5, 0.01), label='Y Error')
ax_hist.hist(zvs_residuals[:, 2], bins=np.arange(-0.5, 0.5, 0.01), label='Z Error')
ax_hist.set_title('Histogram of ZVW Residuals')
ax_hist.set_xlabel('ZVW Residual (g)')
ax_hist.set_ylabel('Count')
ax_hist.grid(True, linestyle='--', alpha=0.5)
ax_hist.legend()
plt.tight_layout()
plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_ZVW_residuals.png'), dpi=120)
# plt.show()


Mean X Error: 0.012098666146324207 g
Mean Y Error: -0.012148534416344718 g
Mean Z Error: 0.02331376303214137 g


In [42]:
# ===========
# ZUPT
# ===========

accel_ms2 = linear_accel * 9.80665

vel_no_zupt = np.zeros_like(accel_ms2)

# =====
# Integrate acceleration to get velocity
# =====

# This should explose because we include ZUPT here
for i in range(1, len(dt_array)):
    vel_no_zupt[i] = vel_no_zupt[i-1] + (accel_ms2[i] * dt_array[i])

# Velocity integration WITH ZUPT
vel_zupt = np.zeros_like(accel_ms2)
for i in range(1, len(dt_array)):
    if (zvw_mask[i]):
        vel_zupt[i] = np.array([0.0,0.0,0.0])
    else:
        vel_zupt[i] = vel_zupt[i-1] + (accel_ms2[i] * dt_array[i])


# =====
# Integrate velocity to get position
# =====
pos_zupt = np.zeros_like(vel_zupt)
for i in range(1, len(dt_array)):
    pos_zupt[i] = pos_zupt[i-1] + (vel_zupt[i] * dt_array[i])

final_xy_distance = np.linalg.norm(pos_zupt[-1, :2])
print(f"Final Calculated XY Distance: {final_xy_distance:.3f} meters")
print(f"Target Distance: 19.985 meters")


# =====
# Plot results
# =====
fig, (ax4, ax5, ax6) = plt.subplots(3, 1, figsize=(14, 15))

# Plot 1: The Explosion (No ZUPT)
ax4.plot(time_sec, vel_no_zupt[:, 0], label='X Velocity', alpha=0.8)
ax4.plot(time_sec, vel_no_zupt[:, 1], label='Y Velocity', alpha=0.8)
ax4.plot(time_sec, vel_no_zupt[:, 2], label='Z Velocity', alpha=0.8)
ax4.set_title('Velocity WITHOUT ZUPT (Drift Explosion)')
ax4.set_ylabel('Velocity (m/s)')
ax4.grid(True, linestyle='--', alpha=0.5)
ax4.legend()

# Plot 2: The Clean Bumps (With ZUPT)
ax5.plot(time_sec, vel_zupt[:, 0], label='X Velocity', alpha=0.8)
ax5.plot(time_sec, vel_zupt[:, 1], label='Y Velocity', alpha=0.8)
ax5.plot(time_sec, vel_zupt[:, 2], label='Z Velocity', alpha=0.8)
ax5.set_title('Velocity WITH ZUPT (Clean Bumps)')
ax5.set_ylabel('Velocity (m/s)')
ax5.grid(True, linestyle='--', alpha=0.5)
ax5.legend()

# Plot 3: 2D Trajectory (Top-Down View)
ax6.plot(pos_zupt[:, 0], pos_zupt[:, 1], label='XY Trajectory', color='tab:green', linewidth=2)
ax6.scatter(pos_zupt[0, 0], pos_zupt[0, 1], color='blue', marker='o', s=100, label='Start')
ax6.scatter(pos_zupt[-1, 0], pos_zupt[-1, 1], color='red', marker='X', s=100, label='End')
ax6.set_title('Calculated 2D Position (Top-Down View)')
ax6.set_xlabel('X Position (m)')
ax6.set_ylabel('Y Position (m)')
ax6.grid(True, linestyle='--', alpha=0.5)
ax6.legend()
ax6.axis('equal') # Keeps the X and Y scale 1:1 so the path isn't warped

plt.tight_layout()
plt.savefig(f'{csv_save_path}{csv_file_name}'.replace('.csv', '_ZUPT_2D_trajectory.png'), dpi=120)
plt.show()

Final Calculated XY Distance: 18.855 meters
Target Distance: 19.985 meters
